# Explainable Boosting Machine — Optuna study on `fe_v0_native` (CPU, ~12 h)

40-trial Optuna study for `interpret`'s `ExplainableBoostingClassifier` (a glass-box
gradient-boosted GA²M) on the **native base features** — the 19 raw columns with **no
feature engineering** (the `fe_v0` baseline).

**Encoding.** EBM consumes categoricals *natively* as nominal features — no one-hot or
ordinal step. We pass the `category`-dtype frame straight through and declare
`feature_types` explicitly (`nominal` for category columns, `continuous` otherwise) so
the binning is reproducible rather than auto-inferred.

**Why CPU?** EBM training is CPU-bound and parallelizes across its outer bags
(`n_jobs=-1`); there is no GPU path. CPU quota is effectively unlimited.

**Settings (right sidebar):** Accelerator → **None / CPU**; Internet → **On**; Add Input →
**playground-series-s6e3**.

> ⏱️ EBM trials are slower than the tree ensembles. **SMOKE-TEST FIRST:** set `n_trials=3`
> to measure per-trial time before the full 40-trial Save & Run All.

In [1]:
# List attached inputs (confirm the competition data is mounted).
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e3/train.csv
/kaggle/input/competitions/playground-series-s6e3/test.csv


In [2]:
import os, sys, subprocess

REPO_URL  = "https://github.com/biswajit-nag/Predict-Customer-Churn.git"
REPO_ROOT = "/kaggle/working/Predict-Customer-Churn"

if not os.path.exists(REPO_ROOT):
    subprocess.run(["git", "clone", REPO_URL, REPO_ROOT], check=True)

os.chdir(REPO_ROOT)                       # CWD = repo root (fixes data paths + git_info)
# Put the clone FIRST on sys.path so its `src` wins over any other module named `src`,
# and drop a possibly-stale `src` cached by an earlier cell.
sys.path.insert(0, REPO_ROOT)
for _m in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
    del sys.modules[_m]
print("CWD:", os.getcwd())

Cloning into '/kaggle/working/Predict-Customer-Churn'...


CWD: /kaggle/working/Predict-Customer-Churn


Updating files: 100% (354/354), done.


In [3]:
# sklearn ships with the image; optuna and interpret (the EBM library) do not.
!pip install -q optuna interpret

import sklearn, optuna, interpret
from interpret.glassbox import ExplainableBoostingClassifier
print("sklearn:  ", sklearn.__version__)
print("optuna:   ", optuna.__version__)
print("interpret:", interpret.__version__)
print("CPU cores:", os.cpu_count())

# Quick sanity fit to confirm the EBM works before the full study.
import numpy as np
_Xs = np.random.rand(2000, 6)
_ys = (np.random.rand(2000) > 0.5).astype(int)
ExplainableBoostingClassifier(max_rounds=50, outer_bags=2, random_state=42).fit(_Xs, _ys)
print("ExplainableBoostingClassifier CPU OK")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 13.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.1/780.1 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.7/270.7 kB 18.1 MB/s eta 0:00:00
sklearn:   1.6.1
optuna:    4.8.0
interpret: 0.7.8
CPU cores: 4
ExplainableBoostingClassifier CPU OK


In [4]:
# data/processed/*.parquet are tracked in the repo, so prepare_data loads them
# directly from the clone (data_hash then matches local runs). Copy the raw CSVs
# as well so prepare_data can rebuild from source if the cache is ever absent.
import shutil
from pathlib import Path

raw_dir = Path(REPO_ROOT) / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
for f in ("train.csv", "test.csv"):
    shutil.copy(f"/kaggle/input/competitions/playground-series-s6e3/{f}", raw_dir / f)

from src.data import prepare_data
train_df, test_df = prepare_data(encoding='native')
print(f'Loaded native: train_df {train_df.shape}, test_df {test_df.shape}')

Loaded from cache (native): train_df (594194, 21), test_df (254655, 20)
Loaded native: train_df (594194, 21), test_df (254655, 20)


In [5]:
import json
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

from src.tracking import DATA_DIR, RUNS_DIR, RUNS_CSV
from src.cv import run_cv_experiment, save_experiment

### Feature setup — native nominal features (`fe_v0_native`)

In [6]:
DATA_VERSION = 'fe_v0_native'

encoded_features = [c for c in train_df.columns if c not in ('id', 'Churn')]
X_train = train_df[encoded_features]
y_train = train_df['Churn']
X_test  = test_df[encoded_features]

# EBM consumes categoricals NATIVELY as nominal features — no one-hot or ordinal step.
# We pass the category-dtype frame straight through and declare feature_types explicitly
# (category-dtype -> 'nominal', everything else -> 'continuous') so the binning is
# reproducible rather than auto-inferred.
feature_types = ['nominal' if isinstance(X_train[c].dtype, pd.CategoricalDtype)
                 else 'continuous' for c in encoded_features]
print(f'X_train {X_train.shape}  X_test {X_test.shape}  features: {len(encoded_features)}')
print(f'nominal: {feature_types.count("nominal")}  continuous: {feature_types.count("continuous")}')

X_train (594194, 19)  X_test (254655, 19)  features: 19
nominal: 15  continuous: 4


### Optuna study — EBM on `fe_v0_native`

In [7]:
import optuna
from interpret.glassbox import ExplainableBoostingClassifier
from sklearn.model_selection import cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

ebm_inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)


def ebm_objective(trial):
    params = {
        'learning_rate':        trial.suggest_float('learning_rate', 0.005, 0.25, log=True),
        'max_bins':             trial.suggest_int('max_bins', 128, 512),
        'max_interaction_bins': trial.suggest_int('max_interaction_bins', 16, 64),
        'interactions':         trial.suggest_int('interactions', 0, 15),
        'min_samples_leaf':     trial.suggest_int('min_samples_leaf', 2, 50),
        'max_leaves':           trial.suggest_int('max_leaves', 2, 5),
        'smoothing_rounds':     trial.suggest_int('smoothing_rounds', 0, 1000),
        'outer_bags':           trial.suggest_int('outer_bags', 4, 12),
        'feature_types':        feature_types,
        'n_jobs':               -1,
        'random_state':         42,
    }
    scores = cross_val_score(
        ExplainableBoostingClassifier(**params), X_train, y_train,
        cv=ebm_inner_cv, scoring='roc_auc',
        n_jobs=1,  # one trial at a time; the EBM bags already use all cores
    )
    return scores.mean()


ebm_study = optuna.create_study(
    study_name='ebm-fe_v0_native',
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
)

# Warm-start from reasonable EBM defaults: no interactions, 8 outer bags.
ebm_study.enqueue_trial({
    'learning_rate':        0.02,
    'max_bins':             256,
    'max_interaction_bins': 32,
    'interactions':         0,
    'min_samples_leaf':     4,
    'max_leaves':           3,
    'smoothing_rounds':     200,
    'outer_bags':           8,
})

# EBM trials are slower than RF/ExtraTrees. SMOKE-TEST with n_trials=3 first to gauge
# per-trial timing, then restore 40 and Save & Run All.
ebm_study.optimize(ebm_objective, n_trials=3, show_progress_bar=True)

print(f'Best inner-CV ROC AUC: {ebm_study.best_value:.6f}  (trial {ebm_study.best_trial.number})')
for k, v in ebm_study.best_params.items():
    print(f'  {k:22s} {v}')

  0%|          | 0/3 [00:00<?, ?it/s]

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/backend/resource_tracker.py", line 326, in main
    registry[rtype][name] -= 1
    ~~~~~~~~~~~~~~~^^^^^^
KeyError: '/dev/shm/joblib_memmapping_folder_16_b7515458aca84e1abb13cb2f62b84ee5_eea9c1061ace4864ac296b485dbe9f1d/16-135707735514992-96ad1a481557448d90ad4124fb7cfa18.pkl'


Best inner-CV ROC AUC: 0.916147  (trial 1)
  learning_rate          0.021642251106469404
  max_bins               494
  max_interaction_bins   51
  interactions           9
  min_samples_leaf       9
  max_leaves             2
  smoothing_rounds       58
  outer_bags             11


### Run configuration

Best Optuna params are reassembled and passed to `run_cv_experiment`, which fits the
5-fold outer CV and prints OOF accuracy + ROC-AUC. `data_version` ties the run back to
the underlying feature parquet.

In [8]:
_ebm_params = dict(ebm_study.best_params)
_ebm_params['feature_types'] = feature_types
_ebm_params['n_jobs']        = -1
_ebm_params['random_state']  = 42

print('Best EBM params:')
for k, v in ebm_study.best_params.items():
    print(f'  {k:22s} {v}')

run_config = {
    'model_factory': lambda params: ExplainableBoostingClassifier(**params),
    'params':        _ebm_params,
    'metric':        accuracy_score,
    'metric_name':   'accuracy',
    'cv':            StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    'tag':           'ebm-optuna-fe_v0_native',
    'notes':         'ExplainableBoostingClassifier (interpret) on fe_v0_native (19 raw native categorical/numeric features, no feature engineering). Categoricals consumed natively as nominal features (feature_types declared explicitly; no one-hot/ordinal). Best params from a 40-trial Optuna study (3-fold inner CV, ROC-AUC, TPE); trial 0 seeded from reasonable EBM defaults. Notebook: kaggle/predict-customer-churn-ebc-cpu-fe_v0.ipynb.',
    'parent_run_id': '',
    'save_models':   False,
    'data_version':  DATA_VERSION,
}

Best EBM params:
  learning_rate          0.021642251106469404
  max_bins               494
  max_interaction_bins   51
  interactions           9
  min_samples_leaf       9
  max_leaves             2
  smoothing_rounds       58
  outer_bags             11


In [9]:
# Step 1 — Run the experiment (fits 5 folds, prints OOF accuracy + ROC-AUC).
result = run_cv_experiment(run_config, X_train, y_train, X_test, encoded_features)

Run ID: 20260612-215050-37f152
Tag:    ebm-optuna-fe_v0_native

Fold 0: accuracy=0.8607  roc_auc=0.9160  (fit 1835.2s)
Fold 1: accuracy=0.8615  roc_auc=0.9170  (fit 1956.3s)


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/backend/resource_tracker.py", line 326, in main
    registry[rtype][name] -= 1
    ~~~~~~~~~~~~~~~^^^^^^
KeyError: '/dev/shm/joblib_memmapping_folder_16_b7515458aca84e1abb13cb2f62b84ee5_da9c939f9c9f4115bd0388b5123e443e/16-135707735514992-96405e19a2344cf0be0d69cc31701d81.pkl'


Fold 2: accuracy=0.8619  roc_auc=0.9165  (fit 1964.2s)
Fold 3: accuracy=0.8628  roc_auc=0.9174  (fit 2001.4s)
Fold 4: accuracy=0.8606  roc_auc=0.9146  (fit 1885.2s)

OOF accuracy: 0.8615
OOF ROC-AUC:  0.9163
Folds:        0.8615 ± 0.0008

Run complete. Call save_experiment(result) to log this run permanently.


In [10]:
# Step 2 — Save the run (review the OOF ROC-AUC above first).
run_id = save_experiment(result)

Saved to: /kaggle/working/Predict-Customer-Churn/experiments/runs/20260612-215050-37f152


### Build a submission (optional)

`test_proba_mean` is the fold-bagged (5 folds) churn probability for the full test
set. The competition metric is ROC-AUC, so submit the probability directly.

In [11]:
submission = pd.DataFrame({
    'id':    test_df['id'],
    'Churn': result['artifacts']['test_proba_mean'],
})
submission.to_csv('/kaggle/working/submission.csv', index=False)
print(submission.head())
print('wrote /kaggle/working/submission.csv', submission.shape)

       id     Churn
0  594194  0.049075
1  594195  0.000683
2  594196  0.098346
3  594197  0.004116
4  594198  0.507239
wrote /kaggle/working/submission.csv (254655, 2)


### Bundle run artifacts + source notebook into one zip

Zips the run directory, `runs.csv`, and the source `.ipynb` into a single archive
on the Output tab. To fold the run back into the local repo, follow
**§7-8 of `docs/kaggle_gpu_workflow.md`**.

In [12]:
import shutil
from pathlib import Path
from src.tracking import RUNS_DIR, RUNS_CSV

BUNDLE = Path('/kaggle/working/bundle')
if BUNDLE.exists():
    shutil.rmtree(BUNDLE)

# 1) heavy run artifacts (params, oof_proba, test_proba_*, metrics, env, git diff)
shutil.copytree(RUNS_DIR / run_id, BUNDLE / 'runs' / run_id)
# 2) the master index row
shutil.copy(RUNS_CSV, BUNDLE / 'runs.csv')
# 3) source notebook committed in the cloned repo
src_nb = Path(REPO_ROOT) / 'kaggle' / 'predict-customer-churn-ebc-cpu-fe_v0.ipynb'
if src_nb.exists():
    shutil.copy(src_nb, BUNDLE / src_nb.name)
    print('bundled notebook:', src_nb.name)
else:
    print('source notebook not found in clone (push it to master first for future runs)')

archive = shutil.make_archive(f'/kaggle/working/{run_id}_bundle', 'zip', BUNDLE)
print('wrote', archive)

source notebook not found in clone (push it to master first for future runs)
wrote /kaggle/working/20260612-215050-37f152_bundle.zip
